# Chapter 7: Regression for health outcomes

**Data Analysis with Python for Health Specialists** | 3 hours

---

## 1. From correlation to prediction

Regression is the workhorse of clinical research: it appears in nearly every epidemiological study, clinical trial analysis, and risk prediction model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

## 2. Simple linear regression

**Question:** How does systolic blood pressure change with age?

In [ ]:
# Simulated clinical data
np.random.seed(42)
n = 200
age = np.random.uniform(25, 80, n)
# True relationship: SBP = 90 + 0.7 * age + noise
sbp = 90 + 0.7 * age + np.random.normal(0, 12, n)

df = pd.DataFrame({"age": age, "systolic_bp": sbp})

# Fit with scipy
slope, intercept, r_value, p_value, std_err = stats.linregress(df["age"], df["systolic_bp"])
print(f"Intercept: {intercept:.2f}")
print(f"Slope: {slope:.2f} mmHg per year of age")
print(f"R-squared: {r_value**2:.3f}")
print(f"p-value: {p_value:.2e}")

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["age"], df["systolic_bp"], alpha=0.5, s=20, color="steelblue")
x_line = np.linspace(25, 80, 100)
ax.plot(x_line, intercept + slope * x_line, color="red", linewidth=2,
        label=f"SBP = {intercept:.1f} + {slope:.2f} x Age")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Systolic BP (mmHg)")
ax.set_title("Simple linear regression: SBP vs Age")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Multiple linear regression

**Question:** Predict systolic BP from age, BMI, and smoking status simultaneously.

In [ ]:
import statsmodels.api as sm

# Expanded simulated data
np.random.seed(42)
n = 300
age = np.random.uniform(25, 80, n)
bmi = np.random.normal(27, 5, n)
smoking = np.random.binomial(1, 0.25, n)  # 25% smokers
# True model: SBP = 70 + 0.6*age + 0.8*BMI + 5*smoking + noise
sbp = 70 + 0.6 * age + 0.8 * bmi + 5 * smoking + np.random.normal(0, 10, n)

df = pd.DataFrame({
    "age": age, "bmi": bmi, "smoking": smoking, "systolic_bp": sbp
})

# Fit multiple regression with statsmodels
X = sm.add_constant(df[["age", "bmi", "smoking"]])
model = sm.OLS(df["systolic_bp"], X).fit()
print(model.summary())

### Interpreting coefficients in clinical context

Each coefficient answers: "holding all other variables constant, what is the expected change in SBP for a one-unit increase in this predictor?"

In [ ]:
# Extract and display coefficients
coef_df = pd.DataFrame({
    "Coefficient": model.params,
    "95% CI lower": model.conf_int()[0],
    "95% CI upper": model.conf_int()[1],
    "p-value": model.pvalues
}).round(4)
print(coef_df)

## 4. Confounders

A confounder is associated with both the exposure and the outcome, distorting the apparent relationship.

In [ ]:
# Example: coffee drinking and heart disease
# Both are associated with smoking (the confounder)
np.random.seed(42)
n = 500
smoking = np.random.binomial(1, 0.3, n)
coffee = 1.5 + 2 * smoking + np.random.normal(0, 1.5, n)
heart_disease_risk = 0.5 * smoking + np.random.normal(0, 0.3, n)

# Unadjusted: coffee appears to predict heart disease
r_unadjusted, p_unadjusted = stats.pearsonr(coffee, heart_disease_risk)
print(f"Unadjusted correlation (coffee vs HD risk): r = {r_unadjusted:.3f}, p = {p_unadjusted:.4f}")

# Adjusted regression: coffee effect disappears
df_conf = pd.DataFrame({"coffee": coffee, "smoking": smoking, "hd_risk": heart_disease_risk})
X = sm.add_constant(df_conf[["coffee", "smoking"]])
model_adj = sm.OLS(df_conf["hd_risk"], X).fit()
print(f"\nAdjusted model:")
print(f"  Coffee coefficient: {model_adj.params['coffee']:.4f} (p = {model_adj.pvalues['coffee']:.4f})")
print(f"  Smoking coefficient: {model_adj.params['smoking']:.4f} (p = {model_adj.pvalues['smoking']:.4f})")

## 5. Checking regression assumptions

In [ ]:
# Residual diagnostics for our SBP model
residuals = model.resid
fitted = model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residuals vs fitted
axes[0].scatter(fitted, residuals, alpha=0.4, s=15)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Fitted")

# Histogram of residuals
axes[1].hist(residuals, bins=25, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Residuals")
axes[1].set_title("Distribution of residuals")

# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title("Q-Q plot")

plt.tight_layout()
plt.show()

## 6. Logistic regression

When the outcome is binary (disease: yes/no), logistic regression models the *probability* of the outcome.

**Question:** Predict whether a patient has diabetes from age, BMI, and glucose level.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Load the Pima Indians Diabetes dataset
url = ("https://raw.githubusercontent.com/jbrownlee/Datasets/"
       "master/pima-indians-diabetes.data.csv")
columns = ["pregnancies", "glucose", "blood_pressure", "skin_thickness",
           "insulin", "bmi", "diabetes_pedigree", "age", "outcome"]
pima = pd.read_csv(url, names=columns)
print(f"Shape: {pima.shape}")
print(f"Diabetes prevalence: {pima['outcome'].mean():.1%}")
pima.head()

In [ ]:
# Fit logistic regression with statsmodels (for detailed output)
X = sm.add_constant(pima[["age", "bmi", "glucose"]])
y = pima["outcome"]

logit_model = sm.Logit(y, X).fit()
print(logit_model.summary())

## 7. Odds ratios from logistic regression

The coefficients of logistic regression are log-odds. Exponentiate them to get **odds ratios**.

In [ ]:
# Odds ratios with 95% confidence intervals
odds_ratios = np.exp(logit_model.params)
ci = np.exp(logit_model.conf_int())

or_df = pd.DataFrame({
    "Odds Ratio": odds_ratios,
    "95% CI lower": ci[0],
    "95% CI upper": ci[1],
    "p-value": logit_model.pvalues
}).round(4)
print(or_df)

## 8. Model evaluation: confusion matrix and beyond

In [ ]:
# Fit with scikit-learn for prediction
features = ["age", "bmi", "glucose"]
X = pima[features]
y = pima["outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print()

# Detailed metrics
print(classification_report(y_test, y_pred,
                            target_names=["No Diabetes", "Diabetes"]))

### Sensitivity, specificity, PPV, NPV

In [ ]:
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)    # true positive rate (recall)
specificity = tn / (tn + fp)    # true negative rate
ppv = tp / (tp + fp)            # positive predictive value (precision)
npv = tn / (tn + fn)            # negative predictive value

print(f"Sensitivity (recall): {sensitivity:.3f}")
print(f"  -> Of those WITH diabetes, {sensitivity:.1%} were correctly identified")
print(f"Specificity:          {specificity:.3f}")
print(f"  -> Of those WITHOUT diabetes, {specificity:.1%} were correctly identified")
print(f"PPV (precision):      {ppv:.3f}")
print(f"  -> Of those PREDICTED positive, {ppv:.1%} actually have diabetes")
print(f"NPV:                  {npv:.3f}")
print(f"  -> Of those PREDICTED negative, {npv:.1%} are truly disease-free")

In [ ]:
# Visualize the confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Diabetes", "Diabetes"],
            yticklabels=["No Diabetes", "Diabetes"], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 9. Introduction to survival analysis

Sometimes the outcome is not "yes/no" but "time until event" --- time to death, relapse, or readmission.

In [ ]:
# !pip install lifelines
from lifelines import KaplanMeierFitter, CoxPHFitter

# Simulated survival data
np.random.seed(42)
n = 200
treatment = np.random.binomial(1, 0.5, n)
time = np.where(treatment == 1,
                np.random.exponential(24, n),
                np.random.exponential(16, n))
event = np.random.binomial(1, 0.75, n)

surv_df = pd.DataFrame({
    "time": time, "event": event, "treatment": treatment
})

In [ ]:
# Kaplan-Meier curves
fig, ax = plt.subplots(figsize=(8, 5))

for label, group_df in surv_df.groupby("treatment"):
    kmf = KaplanMeierFitter()
    kmf.fit(group_df["time"], event_observed=group_df["event"],
            label="Treatment" if label == 1 else "Control")
    kmf.plot_survival_function(ax=ax)

ax.set_xlabel("Time (months)")
ax.set_ylabel("Survival probability")
ax.set_title("Kaplan-Meier survival curves")
plt.tight_layout()
plt.show()

### Cox proportional hazards model

In [ ]:
# Cox proportional hazards
cph = CoxPHFitter()
cph.fit(surv_df, duration_col="time", event_col="event",
        formula="treatment")
cph.print_summary()

In [ ]:
# Hazard ratio
hr = np.exp(cph.params_["treatment"])
print(f"Hazard ratio for treatment: {hr:.3f}")
if hr < 1:
    print(f"Treatment reduces hazard by {(1-hr)*100:.1f}%")
else:
    print(f"Treatment increases hazard by {(hr-1)*100:.1f}%")

## Exercises

Using the Pima Indians Diabetes dataset:

1. **Data preparation.** Replace zero values in glucose, blood_pressure, bmi, skin_thickness, and insulin with NaN. Drop rows with missing values.
2. **Simple logistic regression.** Fit a model predicting diabetes from glucose alone. What is the odds ratio?
3. **Multiple logistic regression.** Fit a model using age, BMI, glucose, blood pressure, and diabetes pedigree. Report odds ratios with 95% CIs.
4. **Model comparison.** Split 70/30. Compare sensitivity, specificity, and accuracy between simple and multiple models.
5. **Confounding.** Fit glucose-only model, then add age. Does the glucose coefficient change?
6. **Clinical thresholds.** Try thresholds 0.3 and 0.7 instead of 0.5. How do sensitivity and specificity change?
7. **Bonus: survival analysis.** Add age and smoking covariates to the Cox model. Which has the largest hazard ratio?

In [ ]:
# Exercise 1: your code here


In [ ]:
# Exercise 2: your code here


In [ ]:
# Exercise 3: your code here


In [ ]:
# Exercise 4: your code here


In [ ]:
# Exercise 5: your code here


In [ ]:
# Exercise 6: your code here


In [ ]:
# Exercise 7: your code here
